# Measure tissue thickness

Variable tissue thickness means a fraction of every FOV's z-stack has no real tissue signal at all -- wasted imaging time. This notebook maps, per FOV, where (in z, µm) real tissue signal actually starts and ends, using one already-finished round (default: `cells`).

Procedure:
1. Resolve the target round's frame table and `CHANNEL_NM`'s z-steps.
2. For every FOV, read every z-plane of `CHANNEL_NM` (still far fewer than the round's full multi-color frame count) and build an EXACT, bin-width-1 histogram of each frame -- a true Counter over observed pixel intensities (`analysis.fov.compute_channel_counters`, stored sparsely via `numpy.unique`, cached per FOV). Exact per-intensity counts mean every later step (reference-frame selection, threshold estimation, the per-z true-pixel-count profile) is derived from this one cached read, without recomputing or re-reading pixels.
3. Across every FOV and z, find the frame with the **highest mean intensity** (best available proxy for "real tissue") and the frame with the **lowest mean intensity** (best available proxy for "background/no tissue") -- not a fixed z pooled across FOVs (an earlier version's approach, which broke down once it became clear some FOVs are blank at that fixed z; see step 4). Display both frames' images, plot both frames' histograms overlaid (log-scale and linear-scale), and derive `THRESHOLD` as the intensity value that best separates the two distributions -- the value minimizing total misclassified pixels between the two Counters (`analysis.fov.two_class_separating_threshold`), review the plot and override `THRESHOLD` manually if it looks wrong.
4. For every FOV, derive its true-pixel-count (NTP) profile directly from its cached Counter (no further disk read). Report the shallowest (`z_first_um`) and deepest (`z_last_um`) z with signal (NTP > `NTP_THRESHOLD`), plus `is_contiguous` (whether signal held continuously in between). Both boundaries matter: some FOVs are blank at the top of the imaged range and only pick up signal partway down, not just "signal that eventually stops".
5. Lay every FOV's `z_first_um`/`z_last_um` out on its stage-position grid and plot as heatmaps.

Figures and results are saved under `SAMPLE_DIR/analysis/figures/` and `SAMPLE_DIR/analysis/` respectively, in addition to being shown inline.

Runs anywhere the standard `SAMPLE_DIR/{data,metadata,positions,analysis}` layout is reachable, including a cluster node (same convention as `05_batch_sample_review.ipynb`/`07_cluster_submit_analysis.ipynb`). Step 2's backfill loop is intentionally sequential, not process-pool-parallelized: on a shared SLURM node, `os.cpu_count()` reports the node's total core count, not this job's actual memory allocation, so sizing a worker pool off it (`config.resolved_n_workers`) can spawn far more workers than the job's real memory allows -- each holding a stack in memory at once -- and get OOM-killed (`BrokenProcessPool`). Reading only `CHANNEL_NM`'s frames (not the whole multi-color stack) keeps the sequential version fast without a pool.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.progress           import ProgressTracker
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import iter_image_frames, path_mtime
from MERci.analysis.fov       import (
    compute_channel_counters, save_channel_counters, load_channel_counters,
    counter_mean, counter_percentile, rebin_counter, ntp_profile_from_counters,
    two_class_separating_threshold,
)
from MERci.acquisition.configs import find_frame_table_for_hal_config, read_hal_exposure_time

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME  = SAMPLE_DIR.name
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Which round to use -- by imaging_type (default "cells"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# A z-plane still counts as "has tissue" if its true-pixel count (NTP) exceeds this.
NTP_THRESHOLD = 1

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Display-histogram resolution (section 5) -- these are re-binned on demand from the
# exact per-intensity Counter, so changing this never requires recomputing anything.
DISPLAY_HIST_BINS      = 200
LINEAR_HIST_PERCENTILE = 99.0   # upper bound of the linear-scale display histogram

# Binarization intensity threshold; None = auto-estimate as the value that best
# separates the highest-mean (tissue) frame's distribution from the lowest-mean
# (background) frame's distribution across every FOV (section 5) -- review that
# plot before trusting the estimate on a new experiment.
THRESHOLD = None

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# This notebook's own cache -- deliberately NOT tracker.histogram_path()'s canonical
# location. FOVScheduler's own per-frame histograms are fixed-width (lossy) 512-bin
# ones, which can't be turned back into an EXACT per-intensity Counter, so unlike
# earlier versions of this notebook there is no shortcut reuse of that canonical
# cache at all here -- every FOV's channel Counters are computed by, and cached
# under, this notebook alone.
tissue_cache_dir     = config.analysis_dir / "tissue_thickness_cache"
channel_counters_dir = tissue_cache_dir / "channel_counters"
channel_counters_dir.mkdir(parents=True, exist_ok=True)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {tissue_cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta):
    """Frame table (columns color/channel/z, 0-based frame-index rows) for round_id's HAL config."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    raise FileNotFoundError(f"No frame table found for round {round_id}")


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))

print(f"Target round : {target_round_id}")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Compute (or load cached) exact per-z Counter histograms for `CHANNEL_NM`

For every FOV, reads every z-plane of `CHANNEL_NM` (still far fewer than the round's
full multi-color frame count) and builds an EXACT, bin-width-1 histogram of each
frame -- a true Counter over observed pixel intensities (`analysis.fov.
compute_channel_counters`, stored sparsely: only intensity values that actually
occur, via `numpy.unique`), not a fixed-bin-count histogram. Having every
intensity's exact count means the reference-frame selection (section 5), the
threshold-estimation display histograms (section 5), and the per-z true-pixel-count
profile (section 6) can ALL be derived from this ONE cached read, without
re-reading pixels or recomputing anything.

No fallback to an existing full per-frame histogram here (unlike earlier versions
of this notebook): `FOVScheduler`'s own histograms are fixed-width (lossy) 512-bin
ones, which can't be turned back into an exact per-intensity Counter -- every FOV's
Counters are computed by, and cached under, this notebook alone
(`analysis/tissue_thickness_cache/channel_counters/`).

In [ ]:
def channel_counters_path(fpath):
    return channel_counters_dir / f"{Path(fpath).stem}_counters.npz"


files = meta.files_for_round(target_round_id)
print(f"Round {target_round_id}: {len(files)} FOV file(s) expected.")

channel_counters = {}   # fov_id -> compute_channel_counters()-shaped dict
from_own_cache, to_compute = [], []
n_missing_on_disk = 0

for fpath in files:
    if channel_counters_path(fpath).exists():
        from_own_cache.append(fpath)
    elif fpath.exists():
        to_compute.append(fpath)
    else:
        n_missing_on_disk += 1

for fpath in from_own_cache:
    channel_counters[meta.fov_id_of_file(fpath)] = load_channel_counters(channel_counters_path(fpath))
print(f"{len(from_own_cache)} channel Counter(s) already cached -- loaded directly.")

n_computed = 0
if to_compute:
    print(f"Computing {len(to_compute)} missing channel Counter(s) sequentially "
          f"(all {len(z_frame_indices)} z-step(s) of channel {CHANNEL_NM:.0f} nm per FOV).")
    reporter = ProgressReporter(total=len(to_compute), label="Computing channel Counters")
    for fpath in reporter.wrap(to_compute):
        counters = compute_channel_counters(
            fpath, z_frame_indices,
            frame_width=config.frame_width, frame_height=config.frame_height,
        )
        save_channel_counters(channel_counters_path(fpath), counters)
        channel_counters[meta.fov_id_of_file(fpath)] = counters
        n_computed += 1

print(f"Channel Counters ready for {len(channel_counters)} / {len(files)} FOVs "
      f"({n_computed} newly computed this run, {n_missing_on_disk} not yet written on disk).")

## 5 — Reference frames (highest- and lowest-mean intensity) and threshold estimation

Across every FOV and every z, finds the frame with the highest mean intensity
(the most tissue-dense frame available anywhere in the round) and the frame
with the lowest mean intensity (the best available proxy for pure background),
rather than pooling one fixed z across every FOV (an earlier version's
approach, which produced a poor, non-bimodal pooled histogram since that fixed
z is blank for some FOVs -- see section 6's `z_first_um`). Displays both
frames' images directly for a visual sanity check, plus their exact
histograms overlaid on log-scale and linear-scale axes. `THRESHOLD` is set to
the intensity value that best separates the two distributions -- the value
minimizing total misclassified pixels across the two Counters
(`analysis.fov.two_class_separating_threshold`, the same criterion behind
Otsu's method, applied directly to two already-labeled distributions instead
of one pooled/unlabeled histogram). Review the plot before trusting it -- if
it looks wrong, set `THRESHOLD` by hand above and re-run from here.

In [ ]:
best  = None   # (mean, fov_id, pos_in_z, frame_idx, z_um) -- highest mean intensity
worst = None   # same shape -- lowest mean intensity
n_frames_considered = 0
for fov_id, counters in channel_counters.items():
    for pos, (values, counts) in enumerate(zip(counters["values_per_z"], counters["counts_per_z"])):
        n_frames_considered += 1
        mean = counter_mean(values, counts)
        if best is None or mean > best[0]:
            best = (mean, fov_id, pos, int(counters["frame_indices"][pos]), float(counters["z_um"][pos]))
        if worst is None or mean < worst[0]:
            worst = (mean, fov_id, pos, int(counters["frame_indices"][pos]), float(counters["z_um"][pos]))

best_mean,  best_fov_id,  best_pos,  best_frame_idx,  best_z_um  = best
worst_mean, worst_fov_id, worst_pos, worst_frame_idx, worst_z_um = worst

best_values,  best_counts  = (channel_counters[best_fov_id]["values_per_z"][best_pos],
                               channel_counters[best_fov_id]["counts_per_z"][best_pos])
worst_values, worst_counts = (channel_counters[worst_fov_id]["values_per_z"][worst_pos],
                               channel_counters[worst_fov_id]["counts_per_z"][worst_pos])

print(f"Highest-mean frame: FOV {best_fov_id}, frame_idx={best_frame_idx}, z={best_z_um:.2f} um "
      f"(mean intensity {best_mean:.0f}, highest of {n_frames_considered} FOV x z combinations)")
print(f"Lowest-mean frame : FOV {worst_fov_id}, frame_idx={worst_frame_idx}, z={worst_z_um:.2f} um "
      f"(mean intensity {worst_mean:.0f}, lowest of {n_frames_considered} FOV x z combinations)")

# Counters have no spatial information -- re-read just these two frames to display them.
best_fpath  = next(f for f in files if meta.fov_id_of_file(f) == best_fov_id)
worst_fpath = next(f for f in files if meta.fov_id_of_file(f) == worst_fov_id)
best_frame  = next(frame for _, frame in iter_image_frames(
    best_fpath, [best_frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
))
worst_frame = next(frame for _, frame in iter_image_frames(
    worst_fpath, [worst_frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
))

fig_img, axes_img = plt.subplots(1, 2, figsize=(10, 5))
for ax, frame, fov_id, z_um, title in zip(
    axes_img, (worst_frame, best_frame), (worst_fov_id, best_fov_id), (worst_z_um, best_z_um),
    ("Lowest-mean frame (background)", "Highest-mean frame (tissue)"),
):
    im = ax.imshow(frame, cmap="gray")
    ax.set_title(f"{title}\nFOV {fov_id}, z={z_um:.1f} um")
    fig_img.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig_img.tight_layout()
fig_img.savefig(figures_dir / f"tissue_thickness_reference_frames_round{target_round_id}.png", dpi=150)
plt.show()

# Threshold that best separates the two labeled distributions (minimizes total
# misclassified pixels between the two exact Counters) -- computed at full
# precision, not from the (lossy, re-binned) display histograms below.
estimated_threshold = two_class_separating_threshold(worst_values, worst_counts, best_values, best_counts)

# Log-scale view (full range) + linear-scale view (combined min -> the HIGH/tissue
# frame's LINEAR_HIST_PERCENTILE-th percentile, since that frame has the
# wider/brighter tail worth showing), both re-binned on demand from the exact
# Counters -- no raw pixel re-read for either.
combined_min = float(min(worst_values.min(), best_values.min()))
combined_max = float(max(worst_values.max(), best_values.max()))
pct_value    = counter_percentile(best_values, best_counts, LINEAR_HIST_PERCENTILE)

log_edges        = np.logspace(np.log10(max(combined_min, 1)), np.log10(combined_max), DISPLAY_HIST_BINS + 1)
log_bin_centers  = np.sqrt(log_edges[:-1] * log_edges[1:])   # geometric mean = correct center in log space
worst_log_counts = rebin_counter(worst_values, worst_counts, log_edges)
best_log_counts  = rebin_counter(best_values,  best_counts,  log_edges)

linear_edges        = np.linspace(combined_min, max(pct_value, combined_min + 1), DISPLAY_HIST_BINS + 1)
linear_bin_centers   = 0.5 * (linear_edges[:-1] + linear_edges[1:])
worst_linear_counts  = rebin_counter(worst_values, worst_counts, linear_edges)
best_linear_counts   = rebin_counter(best_values,  best_counts,  linear_edges)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(log_bin_centers, worst_log_counts, "-", color="steelblue", lw=1.2,
             label=f"lowest-mean frame (FOV {worst_fov_id})")
axes[0].plot(log_bin_centers, best_log_counts, "-", color="darkorange", lw=1.2,
             label=f"highest-mean frame (FOV {best_fov_id})")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)")
axes[0].set_ylabel("Pixel count")
axes[0].set_title("Log-scale (full range)")

axes[1].plot(linear_bin_centers, worst_linear_counts, "-", color="steelblue", lw=1.2,
             label=f"lowest-mean frame (FOV {worst_fov_id})")
axes[1].plot(linear_bin_centers, best_linear_counts, "-", color="darkorange", lw=1.2,
             label=f"highest-mean frame (FOV {best_fov_id})")
axes[1].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)")
axes[1].set_ylabel("Pixel count")
axes[1].set_title(f"Linear-scale (min={combined_min:.0f} -> p{LINEAR_HIST_PERCENTILE:.0f}={pct_value:.0f})")

for ax in axes:
    ax.axvline(estimated_threshold, color="crimson", linestyle="--", lw=1.5,
               label=f"separating threshold = {estimated_threshold:.0f}")
    ax.legend()

fig.suptitle(f"Round {target_round_id} -- lowest- vs. highest-mean frame histograms")
fig.tight_layout()
fig.savefig(figures_dir / f"tissue_thickness_histogram_round{target_round_id}.png", dpi=150)
plt.show()

if THRESHOLD is None:
    THRESHOLD = estimated_threshold
print(f"Using THRESHOLD = {THRESHOLD:.0f}")

## 6 — Per-FOV: true-pixel-count (NTP) profile, derived from cached Counters

Purely in-memory now: every FOV's exact per-z Counter is already cached/loaded from
section 4, so deriving each z's true-pixel count against `THRESHOLD`
(`analysis.fov.ntp_profile_from_counters`) needs no further disk read at all.
Reports both `z_first_um`/`z_last_um` (shallowest/deepest z with signal) and
`is_contiguous` (`False` if signal turned off and back on somewhere in between --
debris, folded tissue, noise) -- some FOVs are blank at the top of the imaged range
and only pick up tissue signal partway down, so both boundaries matter, not just
"signal that eventually stops".

In [ ]:
results = []
for fov_id, counters in channel_counters.items():
    profile = ntp_profile_from_counters(counters, THRESHOLD, NTP_THRESHOLD)
    results.append({
        "fov_id":        fov_id,
        "z_first_um":    profile["z_first_um"],
        "z_last_um":     profile["z_last_um"],
        "is_contiguous": profile["is_contiguous"],
        "x_um":          meta.fovs[fov_id].position[0],
        "y_um":          meta.fovs[fov_id].position[1],
    })

results_df = pd.DataFrame(results)
n_no_signal      = results_df["z_last_um"].isna().sum()
n_not_contiguous = (~results_df["is_contiguous"]).sum()
print(f"{len(results_df)} FOV(s) measured; {n_no_signal} had no z-plane above NTP_THRESHOLD at all; "
      f"{n_not_contiguous} had signal turn off and back on somewhere in between (is_contiguous=False).")
print("z_first_um:")
print(results_df["z_first_um"].describe())
print("z_last_um:")
print(results_df["z_last_um"].describe())

## 7 — Tissue-extent heatmaps across the FOV grid

Two panels sharing one color scale: where tissue signal **starts** (`z_first_um` --
a shallow-imaged range wasted before signal appears would show up here as bright
patches) and where it **ends** (`z_last_um`, the original "how deep does tissue go"
question).

In [ ]:
def positions_to_grid_indices(fov_ids, meta):
    """Stage (x, y) positions -> integer (x_idx, y_idx) grid indices (same approach as
    04_view_intensity_stats.ipynb's heatmap: round to the nearest integer micron, then
    rank each axis's unique values -- robust to float imprecision on a regular grid)."""
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank = {v: i for i, v in enumerate(unique_xs)}
    y_rank = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


fov_ids = results_df["fov_id"].tolist()
grid    = positions_to_grid_indices(fov_ids, meta)
n_x     = max(xi for xi, _ in grid.values()) + 1
n_y     = max(yi for _, yi in grid.values()) + 1


def build_matrix(column):
    matrix = np.full((n_y, n_x), np.nan)
    for _, row in results_df.iterrows():
        xi, yi = grid[row["fov_id"]]
        if pd.notna(row[column]):
            matrix[yi, xi] = row[column]
    return matrix


fig, axes = plt.subplots(1, 2, figsize=(max(9, n_x * 0.7 + 3), max(4, n_y * 0.4 + 1.5)))
for ax, column, title in zip(
    axes,
    ("z_first_um", "z_last_um"),
    ("z_first -- signal starts", "z_last -- signal ends"),
):
    im = ax.imshow(build_matrix(column), cmap="viridis", origin="upper", vmin=0, vmax=MAX_Z_COLORMAP)
    ax.set_title(title)
    ax.set_xlabel("X grid index  (increasing stage X →)")
    ax.set_ylabel("Y grid index  (increasing stage Y ↓)")
cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.04)
cbar.set_label("z (um)")
fig.suptitle(f"Round {target_round_id} -- tissue extent map ({len(fov_ids)} FOVs)")

fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")

## 8 — Experimental vs. theoretical acquisition time per frame

Two ways to estimate how long one frame actually takes: the THEORETICAL rate from
this round's HAL `<exposure_time>` (what `acquisition.dave.estimate_dave_experiment`
would use -- exposure time only, no stage-move/fluidics/readout overhead it doesn't
measure), and the EXPERIMENTAL rate measured directly from real file-write
timestamps -- the time between consecutive FOV files finishing (sorted by actual
write time via `common.io.path_mtime`, robust to on-disk listing order and
zarr-aware), divided by this round's frame count. The experimental rate captures
whatever real overhead the theoretical, exposure-time-only estimate can't see, so
section 9's trim-savings estimate below uses the **experimental** rate, not the
theoretical one.

In [ ]:
files_for_timing = [f for f in meta.files_for_round(target_round_id) if f.exists()]
if len(files_for_timing) < 2:
    raise ValueError(
        f"Need at least 2 written FOV files to measure real inter-FOV acquisition "
        f"timing -- only {len(files_for_timing)} found for round {target_round_id}."
    )

mtimes  = sorted(path_mtime(f) for f in files_for_timing)
delta_s = np.diff(mtimes)   # real wall-clock time between consecutive FOV-movies finishing

# Median, not mean -- robust to the occasional outlier gap (a retry, a brief pause)
# without needing to hand-filter anything.
experimental_delta_s          = float(np.median(delta_s))
experimental_time_per_frame_s = experimental_delta_s / len(frame_table)

exposure_time_s = None
for s in meta.series_for_round(target_round_id):
    if not s.hal_config:
        continue
    exp = read_hal_exposure_time(Path(config.settings_dir) / s.hal_config)
    if exp is not None:
        exposure_time_s = exp
        break
if exposure_time_s is None:
    exposure_time_s = 0.25
    print("WARNING: could not read <exposure_time> from this round's HAL config -- "
          "falling back to 0.25 s/frame (same fallback acquisition.dave.estimate_dave_experiment uses).")
theoretical_time_per_frame_s = exposure_time_s

print(f"Inter-FOV write-time delta (n={len(delta_s)} gaps, {len(files_for_timing)} FOV files): "
      f"median {experimental_delta_s:.2f}s, min {delta_s.min():.2f}s, max {delta_s.max():.2f}s, "
      f"std {delta_s.std():.2f}s")
print(f"Frames/FOV this round: {len(frame_table)}")
print(f"\nTheoretical  (HAL exposure_time only) : {theoretical_time_per_frame_s:.4f} s/frame")
print(f"Experimental (real file-write deltas)  : {experimental_time_per_frame_s:.4f} s/frame")
print(f"Experimental / theoretical ratio       : {experimental_time_per_frame_s / theoretical_time_per_frame_s:.2f}x "
      f"(> 1 means real per-frame time includes overhead the theoretical estimate misses)")
print("\n--> Using the EXPERIMENTAL rate for the time-savings estimate in section 9.")

## 9 — What-if: trim z-range acquisition

If a future acquisition only imaged, per FOV, up to `min(z_last_um + Z_MARGIN_UM,
Z_MAX_TRIMMED_UM)` instead of this round's full z-range, how much less disk space
and acquisition time would the round take? Uses this round's own `frame_table`
(section 3), `results_df` (section 6), and `experimental_time_per_frame_s`
(section 8) directly -- no new image reads.

Every color group in `frame_table` whose z varies (a real focus sweep -- including
any blank/return-to-bead-z frames, since their count can itself depend on total
sweep depth, e.g. `z_return_mode="progressive"`) is assumed to scale with the SAME
per-FOV z cutoff found for `CHANNEL_NM` -- i.e. every channel/color is assumed to
be imaging the same physical tissue volume, so a shallower `CHANNEL_NM` extent
implies a shallower everything-else extent too. Fixed (non-z-swept) frames -- e.g.
a single bead/reference shot -- are unaffected. An FOV with no detected signal at
all (`z_last_um` is `NaN`) is assumed to need 0 z-swept frames (i.e. could be
skipped entirely in the trimmed scheme).

Time savings use `experimental_time_per_frame_s` (section 8 -- measured directly
from real file-write timestamps, so it already includes whatever real per-frame
overhead exists: stage/z-move, fluidics, camera readout, ...) rather than the
theoretical HAL-exposure-time-only rate. `N_ROUNDS_LIKE_THIS` extrapolates this
round's savings to the whole experiment -- NOT auto-assumed (different rounds can
have different color/z-sweep configurations), so it defaults to 1 (this round's
own savings only); set it yourself if you know how many rounds actually share this
z-sweep depth.

In [ ]:
# Extra margin (um) kept beyond each FOV's measured z_last_um, and an absolute cap
# on the trimmed depth regardless of z_last_um.
Z_MARGIN_UM      = 3.0
Z_MAX_TRIMMED_UM = 40.0

# How many rounds of the WHOLE experiment are assumed to share this round's
# z-sweep depth/configuration -- 1 = this round's own savings only. Set explicitly;
# not auto-derived from meta.n_rounds since different rounds can differ.
N_ROUNDS_LIKE_THIS = 1

print(f"Z_MARGIN_UM={Z_MARGIN_UM}, Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM}, "
      f"N_ROUNDS_LIKE_THIS={N_ROUNDS_LIKE_THIS}")

In [ ]:
def format_bytes(n):
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024 or unit == "TiB":
            return f"{n:.2f} {unit}"
        n /= 1024


# Every color group in frame_table, keyed by its (rounded) color -- NaN (blank/
# no-laser) frames get a sentinel key so they form their own group rather than
# being silently dropped by groupby. A group "is z-swept" if its frames actually
# span more than one z value (a real focus sweep); everything else is a fixed,
# unaffected frame (e.g. a single bead/reference shot).
color_key = frame_table["color"].round(0)
color_key = color_key.where(color_key.notna(), -1)

zswept_groups = {}
n_fixed_frames = 0
for key, grp in frame_table.groupby(color_key):
    z_vals = grp["z"].to_numpy()
    if pd.Series(z_vals).nunique() > 1:
        zswept_groups[key] = z_vals
    else:
        n_fixed_frames += len(grp)

n_zswept_frames = sum(len(v) for v in zswept_groups.values())
print(f"Frame table: {len(frame_table)} frame(s)/FOV total -- {len(zswept_groups)} z-swept color "
      f"group(s) ({n_zswept_frames} frame(s)), {n_fixed_frames} fixed frame(s) unaffected by trimming.")


def frames_kept_for_fov(z_needed):
    """Frame count kept under the trimmed scheme: every fixed frame, plus every
    z-swept-group frame at or below z_needed (0 z-swept frames if z_needed is None,
    i.e. no signal was detected in this FOV at all)."""
    if z_needed is None:
        return n_fixed_frames
    return n_fixed_frames + sum(int((z_vals <= z_needed).sum()) for z_vals in zswept_groups.values())


frame_bytes = config.frame_width * config.frame_height * 2   # uint16 -- 2 bytes/pixel

trim_rows = []
for _, row in results_df.iterrows():
    z_last   = row["z_last_um"]
    z_needed = min(z_last + Z_MARGIN_UM, Z_MAX_TRIMMED_UM) if pd.notna(z_last) else None
    n_trimmed = frames_kept_for_fov(z_needed)
    trim_rows.append({
        "fov_id":            row["fov_id"],
        "z_last_um":         z_last,
        "z_needed_um":       z_needed,
        "n_frames_current":  len(frame_table),
        "n_frames_trimmed":  n_trimmed,
        "n_frames_removed":  len(frame_table) - n_trimmed,
    })
trim_df = pd.DataFrame(trim_rows)

# Time savings use the EXPERIMENTAL per-frame rate (section 8), not the theoretical
# exposure-time-only one -- it already reflects whatever real overhead exists.
bytes_saved_per_fov  = trim_df["n_frames_removed"] * frame_bytes
time_saved_per_fov_s = trim_df["n_frames_removed"] * experimental_time_per_frame_s

n_fovs                           = len(trim_df)
total_bytes_current_this_round   = n_fovs * len(frame_table) * frame_bytes
total_time_current_this_round_s  = n_fovs * len(frame_table) * experimental_time_per_frame_s
total_bytes_saved_this_round     = int(bytes_saved_per_fov.sum())
total_time_saved_this_round_s    = float(time_saved_per_fov_s.sum())

print(f"\n--- Round {target_round_id}, {n_fovs} FOV(s) ---")
print(f"Frames/FOV: {len(frame_table)} -> mean {trim_df['n_frames_trimmed'].mean():.1f} "
      f"({trim_df['n_frames_removed'].mean():.1f} removed/FOV on average)")
print(f"Space: {format_bytes(total_bytes_current_this_round)} -> "
      f"{format_bytes(total_bytes_current_this_round - total_bytes_saved_this_round)}  "
      f"(saved {format_bytes(total_bytes_saved_this_round)}, "
      f"{100 * total_bytes_saved_this_round / total_bytes_current_this_round:.1f}%)")
print(f"Time:  {format_duration(total_time_current_this_round_s)} -> "
      f"{format_duration(total_time_current_this_round_s - total_time_saved_this_round_s)}  "
      f"(saved {format_duration(total_time_saved_this_round_s)}, "
      f"{100 * total_time_saved_this_round_s / total_time_current_this_round_s:.1f}%)  "
      f"[using experimental_time_per_frame_s={experimental_time_per_frame_s:.4f}s/frame -- section 8]")

if N_ROUNDS_LIKE_THIS != 1:
    print(f"\n--- Extrapolated to {N_ROUNDS_LIKE_THIS} round(s) assumed to share this z-sweep "
          f"(N_ROUNDS_LIKE_THIS) ---")
    print(f"Space saved: {format_bytes(total_bytes_saved_this_round * N_ROUNDS_LIKE_THIS)}")
    print(f"Time saved:  {format_duration(total_time_saved_this_round_s * N_ROUNDS_LIKE_THIS)}")

trim_csv = config.analysis_dir / f"tissue_thickness_zrange_trim_round{target_round_id}.csv"
trim_df.to_csv(trim_csv, index=False)
print(f"\nSaved: {trim_csv}")